# 🔗 Prompt Chaining Design Patterns

**Agentic Design Pattern #1 — Sequential Reasoning Pipelines**

This notebook demonstrates 5 core prompt-chaining techniques using **LangChain** + **Azure OpenAI**:

| # | Pattern | Key Idea |
|---|---------|----------|
| 1 | **Sequential Reasoning Pipeline** | Feed output of step N as input to step N+1 |
| 2 | **Step Decomposition** | Break a complex task into explicit subtasks |
| 3 | **Chain-of-Thought (CoT)** | "Think step by step" to improve reasoning |
| 4 | **Self-Consistency Prompting** | Sample multiple reasoning paths, pick the majority answer |
| 5 | **Program-of-Thought (PoT)** | Generate *code* instead of natural-language reasoning, then execute it |

---

## 0 · Setup & Configuration

In [2]:
import os
import re
import json
from collections import Counter

from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# ── Azure OpenAI LLM ──────────────────────────────────────────
llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.3,
)

parser = StrOutputParser()

print("✅ LLM ready:", llm.deployment_name)

✅ LLM ready: gpt-4o-mini


---
## 1 · Sequential Reasoning Pipeline

The simplest form of prompt chaining: **pipe the output of one LLM call into the next**.  
Each step has a focused role — research → outline → write — so the model does one thing well at each stage.

```
Topic ──▶ [Research] ──▶ [Outline] ──▶ [Draft] ──▶ Final Article
```

In [5]:
research_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research analyst. Given a topic, list 5 key facts or insights about it. Be concise."),
    ("human", "Topic: {topic}")
])

outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a content strategist. Given research notes, create a structured blog-post outline with sections and bullet points."),
    ("human", "Research Notes:\n{research}")
])

draft_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a skilled writer. Given an outline, write a short, engaging blog post (~200 words). Use clear language."),
    ("human", "Outline:\n{outline}")
])

sequential_chain = (
    research_prompt | llm | parser                 
    | (lambda research: {"research": research})   
    | outline_prompt | llm | parser                
    | (lambda outline: {"outline": outline})       
    | draft_prompt | llm | parser                  
)

result = sequential_chain.invoke({"topic": "The impact of AI in movies."})
print(result)

Artificial intelligence is rapidly transforming the film industry, reshaping how movies are made, marketed, and enjoyed. From pre-production to post-production, AI is enhancing creativity and efficiency in unprecedented ways.

One of the most striking advances is in visual effects. AI-powered tools create stunningly realistic CGI, cutting down production time and costs. Films leveraging these technologies deliver breathtaking visuals that captivate audiences like never before. Beyond visuals, AI assists scriptwriters by generating plot ideas, crafting dialogue, and analyzing characters, helping writers overcome creative blocks and enrich storytelling.

Personalization is another game-changer. AI analyzes viewer preferences to recommend movies tailored to individual tastes, while studios use these insights to design targeted marketing campaigns that reach the right audiences effectively. Meanwhile, deepfake technology enables digital actors and de-aging effects, opening new storytelling

---
## 2 · Step Decomposition (Task → Subtasks)

Instead of hard-coding the pipeline, we ask the LLM itself to **decompose a complex task into subtasks**, then execute each subtask sequentially.  
This is useful when you don't know the steps in advance.

```
Complex Task ──▶ [Decompose] ──▶ Subtask 1 ──▶ Subtask 2 ──▶ … ──▶ [Synthesize] ──▶ Final Answer
```

In [6]:
# ── Step 1: Decompose the task into subtasks ──────────────────
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a task-planning assistant.\n"
     "Given a complex task, break it down into 3-5 independent subtasks.\n"
     "Return ONLY a JSON array of strings, e.g.:\n"
     '["subtask 1", "subtask 2", "subtask 3"]'),
    ("human", "Task: {task}")
])

# ── Step 2: Execute each subtask ─────────────────────────────
subtask_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Complete the following subtask thoroughly in 2-3 sentences."),
    ("human", "Subtask: {subtask}")
])

# ── Step 3: Synthesize results ───────────────────────────────
synthesize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a synthesis expert. Given results from multiple subtasks, "
     "combine them into a single coherent answer."),
    ("human", "Original Task: {task}\n\nSubtask Results:\n{results}")
])


def step_decomposition(task: str) -> str:
    """Dynamically decompose → execute → synthesize."""
    # 1. Decompose
    decompose_chain = decompose_prompt | llm | parser
    raw_subtasks = decompose_chain.invoke({"task": task})
    subtasks = json.loads(raw_subtasks)
    print(f"📋 Decomposed into {len(subtasks)} subtasks:")
    for i, st in enumerate(subtasks, 1):
        print(f"   {i}. {st}")

    # 2. Execute each subtask
    subtask_chain = subtask_prompt | llm | parser
    results = []
    for i, st in enumerate(subtasks, 1):
        res = subtask_chain.invoke({"subtask": st})
        results.append(f"Subtask {i} ({st}):\n{res}")
        print(f"   ✅ Subtask {i} done")

    # 3. Synthesize
    synth_chain = synthesize_prompt | llm | parser
    final = synth_chain.invoke({
        "task": task,
        "results": "\n\n".join(results)
    })
    return final


task = "Compare Python and Rust for building a high-performance web API and recommend one with justification."
answer = step_decomposition(task)
print("\n" + "═" * 60)
print("🎯 FINAL ANSWER:\n")
print(answer)

📋 Decomposed into 5 subtasks:
   1. Research the performance characteristics and concurrency models of Python and Rust
   2. Analyze the ecosystem and libraries available for building high-performance web APIs in both languages
   3. Evaluate developer productivity, ease of use, and community support for Python and Rust in web API development
   4. Compare real-world benchmarks and case studies of web APIs built with Python and Rust
   5. Synthesize findings and recommend the most suitable language with justification
   ✅ Subtask 1 done
   ✅ Subtask 2 done
   ✅ Subtask 3 done
   ✅ Subtask 4 done
   ✅ Subtask 5 done

════════════════════════════════════════════════════════════
🎯 FINAL ANSWER:

When comparing Python and Rust for building a high-performance web API, several key factors emerge from their performance characteristics, ecosystems, developer experience, and real-world usage:

1. **Performance and Concurrency**:  
   Rust offers superior runtime performance and concurrency capa

---
## 3 · Chain-of-Thought (CoT) Prompting

We explicitly instruct the model to **think step by step** before giving a final answer.  
This dramatically improves performance on reasoning, math, and logic tasks.

We show two variants:
- **Zero-shot CoT** — just add "think step by step"
- **Few-shot CoT** — provide examples of step-by-step reasoning

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  3a · Zero-Shot Chain-of-Thought
# ═══════════════════════════════════════════════════════════════

zero_shot_cot_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful math tutor.\n"
     "When solving a problem, ALWAYS think step by step.\n"
     "Show your reasoning, then give the final answer on the last line as:\n"
     "ANSWER: <number>"),
    ("human", "{question}")
])

cot_chain = zero_shot_cot_prompt | llm | parser

question = (
    "A store sells apples for $2 each and oranges for $3 each. "
    "Maria buys 4 apples and 5 oranges. She pays with a $50 bill. "
    "How much change does she receive?"
)

print("🧠 Zero-Shot CoT:\n")
print(cot_chain.invoke({"question": question}))

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  3b · Few-Shot Chain-of-Thought
# ═══════════════════════════════════════════════════════════════

few_shot_cot_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a math problem solver. Follow the demonstrated reasoning style."),
    ("human",
     "Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
     "Each can has 3 tennis balls. How many tennis balls does he have now?"),
    ("assistant",
     "Step 1: Roger starts with 5 tennis balls.\n"
     "Step 2: He buys 2 cans × 3 balls/can = 6 new balls.\n"
     "Step 3: Total = 5 + 6 = 11 tennis balls.\n"
     "ANSWER: 11"),
    ("human",
     "Q: A cafeteria had 23 apples. They used 20 to make lunch and bought 6 more. "
     "How many apples do they have?"),
    ("assistant",
     "Step 1: Start with 23 apples.\n"
     "Step 2: Used 20 → 23 - 20 = 3 apples left.\n"
     "Step 3: Bought 6 more → 3 + 6 = 9 apples.\n"
     "ANSWER: 9"),
    ("human", "Q: {question}")
])

few_shot_chain = few_shot_cot_prompt | llm | parser

question2 = (
    "A farmer has 3 fields. Each field has 12 rows of corn. "
    "Each row produces 8 ears of corn. If the farmer sells each ear for $0.50, "
    "how much total revenue does the farmer earn?"
)

print("🧠 Few-Shot CoT:\n")
print(few_shot_chain.invoke({"question": question2}))

---
## 4 · Self-Consistency Prompting

Instead of relying on a single CoT path, we **sample multiple independent reasoning paths** (using higher temperature) and select the answer that appears most frequently — a **majority-vote** ensemble.

```
Question ──▶ [CoT path 1] ──▶ Answer A
         ──▶ [CoT path 2] ──▶ Answer B
         ──▶ [CoT path 3] ──▶ Answer A
         ──▶ [CoT path 4] ──▶ Answer A
         ──▶ [CoT path 5] ──▶ Answer B
                                        ──▶ Majority Vote: Answer A ✓
```

In [ ]:
# ── High-temperature LLM for diverse reasoning paths ─────────
llm_diverse = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0.9,   # ← high temp for diverse reasoning
)

sc_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a math problem solver.\n"
     "Think step by step to solve the problem.\n"
     "On the very last line, write ONLY: ANSWER: <number>"),
    ("human", "{question}")
])

sc_chain = sc_prompt | llm_diverse | parser


def extract_answer(text: str) -> str:
    """Pull the final ANSWER: <value> from the response."""
    match = re.search(r"ANSWER:\s*(.+?)\s*$", text, re.MULTILINE | re.IGNORECASE)
    return match.group(1).strip() if match else text.strip().split("\n")[-1]


def self_consistency(question: str, n_samples: int = 5) -> str:
    """Run n_samples independent CoT paths and majority-vote."""
    answers = []
    for i in range(n_samples):
        response = sc_chain.invoke({"question": question})
        ans = extract_answer(response)
        answers.append(ans)
        print(f"  Path {i+1}: {ans}")

    # Majority vote
    vote = Counter(answers).most_common(1)[0]
    print(f"\n🗳️  Majority vote: {vote[0]}  ({vote[1]}/{n_samples} paths)")
    return vote[0]


question3 = (
    "A train travels from City A to City B at 60 km/h and returns at 40 km/h. "
    "The distance between the cities is 120 km. "
    "What is the average speed for the entire round trip in km/h?"
)

print("🔄 Self-Consistency Prompting:\n")
final_answer = self_consistency(question3, n_samples=5)
print(f"\n✅ Final answer: {final_answer}")

---
## 5 · Program-of-Thought (PoT)

Instead of reasoning in natural language, the LLM generates **executable Python code** to solve the problem.  
We then run that code locally to get a precise, verifiable answer — eliminating arithmetic errors.

```
Question ──▶ [LLM generates Python code] ──▶ [Execute code] ──▶ Precise Answer
```

> ⚠️ **Safety note**: We use a restricted `exec()` with a limited namespace. In production, use a sandbox.

In [ ]:
pot_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Python programmer who solves math/logic problems by writing code.\n\n"
     "Rules:\n"
     "1. Write a Python function called `solve()` that computes the answer.\n"
     "2. The function must RETURN the final numeric answer.\n"
     "3. Return ONLY the code inside a ```python ... ``` block. No explanation.\n"
     "4. Use only built-in Python — no imports."),
    ("human", "{question}")
])

pot_chain = pot_prompt | llm | parser


def extract_code(text: str) -> str:
    """Extract Python code from a markdown code block."""
    match = re.search(r"```(?:python)?\s*\n(.+?)\n```", text, re.DOTALL)
    return match.group(1) if match else text


def program_of_thought(question: str):
    """Generate code → extract → execute → return answer."""
    # 1. Generate code
    raw = pot_chain.invoke({"question": question})
    code = extract_code(raw)
    print("📝 Generated Code:")
    print("─" * 40)
    print(code)
    print("─" * 40)

    # 2. Execute in a restricted namespace
    namespace = {}
    exec(code, {"__builtins__": {"range": range, "len": len, "sum": sum,
                                 "min": min, "max": max, "abs": abs,
                                 "round": round, "int": int, "float": float,
                                 "print": print, "enumerate": enumerate,
                                 "zip": zip, "map": map, "list": list}},
         namespace)

    # 3. Call solve()
    answer = namespace["solve"]()
    return answer


question4 = (
    "A rectangular garden has a perimeter of 56 meters. "
    "Its length is 4 meters more than twice its width. "
    "What is the area of the garden in square meters?"
)

print("💻 Program-of-Thought:\n")
answer = program_of_thought(question4)
print(f"\n✅ Computed Answer: {answer}")

In [ ]:
# ── Bonus: PoT vs CoT comparison on the same question ────────

comparison_question = (
    "A company has 250 employees. 40% work in engineering, 25% in sales, "
    "and the rest in operations. Engineering gets a 10% bonus, sales gets 15%, "
    "and operations gets 5%. If the base salary is $5000/month, "
    "what is the total monthly payroll including bonuses?"
)

print("═" * 60)
print("COMPARISON: PoT vs CoT on the same problem")
print("═" * 60)

# ── CoT answer ────────────────────────────────────────────────
print("\n🧠 Chain-of-Thought:")
cot_answer = cot_chain.invoke({"question": comparison_question})
print(cot_answer)

# ── PoT answer ────────────────────────────────────────────────
print("\n💻 Program-of-Thought:")
pot_answer = program_of_thought(comparison_question)
print(f"\n✅ PoT Computed Answer: {pot_answer}")

---
## 📊 Summary

| Pattern | When to Use | Strength |
|---------|------------|----------|
| **Sequential Pipeline** | Multi-stage content generation (research → write) | Clean separation of concerns |
| **Step Decomposition** | Complex tasks where subtasks aren't known upfront | Dynamic planning |
| **Chain-of-Thought** | Math, logic, and reasoning problems | Improved accuracy via explicit reasoning |
| **Self-Consistency** | Hard problems where a single CoT path may err | Ensemble reliability |
| **Program-of-Thought** | Numerical or algorithmic problems | Exact computation, no arithmetic errors |

### Key Takeaways
- **Chaining** = composing focused prompts into pipelines (LangChain's `|` operator).
- **CoT** improves reasoning by making the model "show its work".
- **Self-Consistency** adds robustness by sampling multiple paths.
- **PoT** offloads computation to code — best for precise numerical answers.

e:\agentic design pattern\Prompt Chaining\.venv\Scripts\python.exe
